In [16]:
!pip install dash pandas plotly

In [17]:
# Tugas Akhir - Bagian 2: Dashboard Statistik Penjualan Mobil dengan Plotly & Dash
# Jalankan:  python3 Final_Assignment_Part2_Dashboard.py   (lalu buka http://127.0.0.1:8050)

import pandas as pd
import dash
from dash import dcc, html
from dash.dependencies import Input, Output
import plotly.express as px

In [18]:
# ----------------------------------------------------------------------------
# 1. Muat data
# ----------------------------------------------------------------------------
URL = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-DV0101EN-SkillsNetwork/Data%20Files/historical_automobile_sales.csv"
data = pd.read_csv(URL)

In [19]:
(data)

,Date,Year,Month,Recession,Consumer_Confidence,Seasonality_Weight,Price,Advertising_Expenditure,Competition,GDP,Growth_Rate,unemployment_rate,Automobile_Sales,Vehicle_Type,City
0,1/31/1980,1980,Jan,1,108.24,0.50,27483.571,1558,7,60.223,0.010000,5.4,456.0,Supperminicar,Georgia
1,2/29/1980,1980,Feb,1,98.75,0.75,24308.678,3048,4,45.986,-0.309594,4.8,555.9,Supperminicar,New York
2,3/31/1980,1980,Mar,1,107.48,0.20,28238.443,3137,3,35.141,-0.308614,3.4,620.0,Mediumfamilycar,New York
3,4/30/1980,1980,Apr,1,115.01,1.00,32615.149,1653,7,45.673,0.230596,4.2,702.8,Supperminicar,Illinois
4,5/31/1980,1980,May,1,98.72,0.20,23829.233,1319,4,52.997,0.138197,5.3,770.4,Smallfamiliycar,California
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
523,8/31/2023,2023,Aug,0,103.36,0.25,27852.993,1793,6,57.169,0.764155,2.6,1579.6,Executivecar,New York
524,9/30/2023,2023,Sep,0,101.55,0.07,21183.704,1028,5,59.315,0.036180,2.5,1123.4,Smallfamiliycar,California
525,10/31/2023,2023,Oct,0,124.66,0.12,15975.589,1148,9,19.472,-2.046169,2.5,1685.9,Sports,California
526,11/30/2023,2023,Nov,0,97.09,0.25,16862.288,4850,5,27.904,0.302179,2.9,2124.6,Smallfamiliycar,Georgia


In [20]:
# Samakan nama kolom pengangguran (beberapa versi dataset memakai huruf besar)
data.columns = data.columns.str.strip()
data = data.rename(columns={c: "unemployment_rate" for c in data.columns if c.lower() == "unemployment_rate"})

year_list = sorted(data["Year"].unique())
month_order = data["Month"].drop_duplicates().tolist()   # urutan kalender sesuai data

In [21]:
# ----------------------------------------------------------------------------
# 2. Buat aplikasi Dash (TODO 4.1) - judul yang bermakna
# ----------------------------------------------------------------------------
app = dash.Dash(__name__)
app.config.suppress_callback_exceptions = True

app.title = "Automobile Sales Statistics Dashboard"

In [22]:
# ----------------------------------------------------------------------------
# 3. Layout: judul, menu drop-down, dan divisi output (TODO 4.1 - 4.3)
# ----------------------------------------------------------------------------
dropdown_options = [
    {"label": "Yearly Statistics", "value": "Yearly Statistics"},
    {"label": "Recession Period Statistics", "value": "Recession Period Statistics"},
]

app.layout = html.Div([
    # TODO 4.1 - judul dashboard
    html.H1(
        "Automobile Sales Statistics Dashboard",
        style={"textAlign": "center", "color": "#503D36", "fontSize": 28, "marginBottom": "20px"},
    ),

    # TODO 4.2 - menu drop-down untuk jenis laporan
    html.Div([
        html.Label("Select Statistics:", style={"fontWeight": "bold"}),
        dcc.Dropdown(
            id="dropdown-statistics",
            options=dropdown_options,
            placeholder="Select a report type",
            value=None,
            style={"width": "80%", "padding": "3px", "fontSize": "18px", "textAlignLast": "center"},
        ),
    ], style={"margin": "10px 0"}),

    # TODO 4.2 - menu drop-down untuk tahun (aktif hanya pada Yearly Statistics)
    html.Div([
        html.Label("Select Year:", style={"fontWeight": "bold"}),
        dcc.Dropdown(
            id="select-year",
            options=[{"label": int(i), "value": int(i)} for i in year_list],
            placeholder="Select a year",
            value=int(year_list[0]),
            style={"width": "80%", "padding": "3px", "fontSize": "18px", "textAlignLast": "center"},
        ),
    ], style={"margin": "10px 0"}),

    # TODO 4.3 - divisi untuk tampilan output (dengan id dan className)
    html.Div(
        id="output-container",
        className="chart-grid",
        style={"display": "flex", "flexDirection": "column"},
    ),
])



In [23]:
# ----------------------------------------------------------------------------
# 4. Callback pertama (TODO 4.4): mengaktifkan / menonaktifkan dropdown tahun
# ----------------------------------------------------------------------------
@app.callback(
    Output(component_id="select-year", component_property="disabled"),
    Input(component_id="dropdown-statistics", component_property="value"),
)
def update_input_container(selected_statistics):
    # Dropdown tahun hanya aktif untuk "Yearly Statistics"
    return selected_statistics != "Yearly Statistics"


def chart_box(figure):
    """Bungkus grafik agar dua grafik dapat berdampingan dalam satu baris."""
    return html.Div(dcc.Graph(figure=figure), style={"width": "50%"})



In [24]:
# ----------------------------------------------------------------------------
# 5. Callback kedua (TODO 4.4 - 4.6): memperbarui kontainer output
# ----------------------------------------------------------------------------
@app.callback(
    Output(component_id="output-container", component_property="children"),
    [Input(component_id="dropdown-statistics", component_property="value"),
     Input(component_id="select-year", component_property="value")],
)
def update_output_container(selected_statistics, input_year):

    # ---------------- TODO 4.5: Recession Report Statistics ----------------
    if selected_statistics == "Recession Period Statistics":
        recession_data = data[data["Recession"] == 1]

        # Plot 1: fluktuasi rata-rata penjualan per tahun selama resesi
        yearly_rec = recession_data.groupby("Year")["Automobile_Sales"].mean().reset_index()
        R_chart1 = px.line(
            yearly_rec, x="Year", y="Automobile_Sales", markers=True,
            title="Average Automobile Sales Fluctuation over Recession Period",
            labels={"Automobile_Sales": "Average Automobile Sales"},
        )

        # Plot 2: rata-rata kendaraan terjual per jenis kendaraan
        average_sales = recession_data.groupby("Vehicle_Type")["Automobile_Sales"].mean().reset_index()
        R_chart2 = px.bar(
            average_sales, x="Vehicle_Type", y="Automobile_Sales", color="Vehicle_Type",
            title="Average Number of Vehicles Sold by Vehicle Type during Recession",
            labels={"Automobile_Sales": "Average Automobile Sales", "Vehicle_Type": "Vehicle Type"},
        )

        # Plot 3: porsi pengeluaran iklan per jenis kendaraan selama resesi
        exp_rec = recession_data.groupby("Vehicle_Type")["Advertising_Expenditure"].sum().reset_index()
        R_chart3 = px.pie(
            exp_rec, values="Advertising_Expenditure", names="Vehicle_Type",
            title="Total Advertising Expenditure Share by Vehicle Type during Recession",
        )

        # Plot 4: pengaruh tingkat pengangguran terhadap penjualan per jenis kendaraan
        unemp_data = (recession_data.groupby(["unemployment_rate", "Vehicle_Type"])["Automobile_Sales"]
                      .mean().reset_index())
        R_chart4 = px.bar(
            unemp_data, x="unemployment_rate", y="Automobile_Sales", color="Vehicle_Type",
            title="Effect of Unemployment Rate on Vehicle Type and Sales",
            labels={"unemployment_rate": "Unemployment Rate", "Automobile_Sales": "Average Automobile Sales"},
        )

        return [
            html.Div(className="chart-item", children=[chart_box(R_chart1), chart_box(R_chart2)],
                     style={"display": "flex"}),
            html.Div(className="chart-item", children=[chart_box(R_chart3), chart_box(R_chart4)],
                     style={"display": "flex"}),
        ]

    # ---------------- TODO 4.6: Yearly Report Statistics ----------------
    elif selected_statistics == "Yearly Statistics" and input_year:
        yearly_data = data[data["Year"] == input_year]

        # Plot 1: rata-rata penjualan tahunan sepanjang periode data
        yas = data.groupby("Year")["Automobile_Sales"].mean().reset_index()
        Y_chart1 = px.line(
            yas, x="Year", y="Automobile_Sales", markers=True,
            title="Yearly Average Automobile Sales (Whole Period)",
            labels={"Automobile_Sales": "Average Automobile Sales"},
        )

        # Plot 2: total penjualan bulanan pada tahun terpilih
        mas = yearly_data.groupby("Month")["Automobile_Sales"].sum().reset_index()
        Y_chart2 = px.line(
            mas, x="Month", y="Automobile_Sales", markers=True,
            category_orders={"Month": month_order},
            title=f"Total Monthly Automobile Sales in {input_year}",
            labels={"Automobile_Sales": "Total Automobile Sales"},
        )

        # Plot 3: rata-rata kendaraan terjual per jenis kendaraan pada tahun terpilih
        avr_vdata = yearly_data.groupby("Vehicle_Type")["Automobile_Sales"].mean().reset_index()
        Y_chart3 = px.bar(
            avr_vdata, x="Vehicle_Type", y="Automobile_Sales", color="Vehicle_Type",
            title=f"Average Vehicles Sold by Vehicle Type in {input_year}",
            labels={"Automobile_Sales": "Average Automobile Sales", "Vehicle_Type": "Vehicle Type"},
        )

        # Plot 4: total pengeluaran iklan per jenis kendaraan pada tahun terpilih
        exp_data = yearly_data.groupby("Vehicle_Type")["Advertising_Expenditure"].sum().reset_index()
        Y_chart4 = px.pie(
            exp_data, values="Advertising_Expenditure", names="Vehicle_Type",
            title=f"Total Advertisement Expenditure for Each Vehicle Type in {input_year}",
        )

        return [
            html.Div(className="chart-item", children=[chart_box(Y_chart1), chart_box(Y_chart2)],
                     style={"display": "flex"}),
            html.Div(className="chart-item", children=[chart_box(Y_chart3), chart_box(Y_chart4)],
                     style={"display": "flex"}),
        ]

    # Belum ada pilihan yang valid
    return html.Div("Please select a report type from the drop-down menu.",
                    style={"textAlign": "center", "padding": "30px", "color": "gray"})


# ----------------------------------------------------------------------------
# 6. Jalankan server
# ----------------------------------------------------------------------------
if __name__ == "__main__":
    app.run(jupyter_mode="inline", debug=False)

<IPython.core.display.Javascript object>